## Load Data

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

job_folder = Path("../../data/job")

def clean_html(text):
    if not text:
        return ""
    text = str(text)
    soup = BeautifulSoup(text, "html.parser")
    cleaned = soup.get_text(separator=" ")
    cleaned = cleaned.replace("\xa0", " ")
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

rows = []

for file_path in job_folder.glob("*.json"):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            job = json.load(f)

        skills_list = [
            s.get("skill")
            for s in job.get("skills", [])
            if isinstance(s, dict) and s.get("skill")
        ]

        flexible_work_arrangements = [
            fwa.get("flexibleWorkArrangement") or fwa.get("arrangement") or fwa.get("name")
            for fwa in job.get("flexibleWorkArrangements", [])
            if isinstance(fwa, dict)
        ]

        employment_types = [
            e.get("employmentType")
            for e in job.get("employmentTypes", [])
            if isinstance(e, dict) and e.get("employmentType")
        ]

        position_levels = [
            p.get("position")
            for p in job.get("positionLevels", [])
            if isinstance(p, dict) and p.get("position")
        ]

        salary_info = job.get("salary") or {}
        salary_type = (salary_info.get("type") or {}).get("salaryType")

        metadata = job.get("metadata") or {}

        row = {
            "uuid": job.get("uuid"),
            "title": job.get("title"),
            "description": clean_html(job.get("description")),
            "minimum_years_experience": job.get("minimumYearsExperience"),
            "skills": skills_list,

            "ssoc_code": job.get("ssocCode"),
            "ssoc_version": job.get("ssocVersion"),

            "salary_minimum": salary_info.get("minimum"),
            "salary_maximum": salary_info.get("maximum"),
            "salary_type": salary_type,

            "employment_types": employment_types,
            "position_levels": position_levels,
            "flexible_work_arrangements": flexible_work_arrangements,

            "posting_date": metadata.get("newPostingDate") or metadata.get("originalPostingDate"),
            "expiry_date": metadata.get("expiryDate"),
        }

        rows.append(row)

    except Exception as e:
        print(f"Error reading {file_path.name}: {e}")



original_jobs_df = pd.DataFrame(rows)

print("Original shape:", original_jobs_df.shape)
original_jobs_df.head()

Original shape: (22720, 15)


# Cleaning

## Create copy of jobs_df

In [405]:
jobs_df = original_jobs_df.copy()
jobs_df.head(1)

,uuid,title,description,minimum_years_experience,skills,ssoc_code,ssoc_version,salary_minimum,salary_maximum,salary_type,employment_types,position_levels,flexible_work_arrangements,posting_date,expiry_date
0,2cc5117f205f59b400cb529d2253651a,Outdoor Sales Engineer (Own Car) | Petrochemic...,Role: Outdoor Sales Engineer Working Hours: 8....,1,"[Project Bidding, Technical Demonstrations, Sh...",24331,2020v2,3000,6000,Monthly,"[Permanent, Full Time]",[Executive],[],2026-01-29,2026-02-28


In [406]:
# Only keep years 0 and 1
jobs_df = jobs_df[jobs_df["minimum_years_experience"].isin([0, 1])]

## Removing interns/ further cleaning

In [407]:
# -------------------------
# 1. lowercase
# -------------------------
jobs_df["title_clean"] = jobs_df["title"].str.lower()
jobs_df["description_clean"] = jobs_df["description"].str.lower()


# -------------------------
# 2. internship filter
# -------------------------
intern_mask = (
    # title / description
    jobs_df["title_clean"].str.contains(r"\bintern(ship)?s?\b", regex=True, na=False) |
    jobs_df["description_clean"].str.contains(r"\bintern(ship)?s?\b", regex=True, na=False)
)


# -------------------------
# 3. employment_types filter
# -------------------------
employment_mask = jobs_df["employment_types"].apply(
    lambda x: isinstance(x, list) and any(
        "intern" in str(e).lower() for e in x
    )
)


# -------------------------
# 4. combine + filter
# -------------------------
jobs_df = jobs_df[~(intern_mask | employment_mask)]

/var/folders/gn/g9t9ks6j5qq32pcsbtb27jxr0000gn/T/ipykernel_2597/3693494163.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  jobs_df["title_clean"].str.contains(r"\bintern(ship)?s?\b", regex=True, na=False) |
/var/folders/gn/g9t9ks6j5qq32pcsbtb27jxr0000gn/T/ipykernel_2597/3693494163.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  jobs_df["description_clean"].str.contains(r"\bintern(ship)?s?\b", regex=True, na=False)


In [408]:
# Create a True/ False for Flexible Work Arrangments
jobs_df["has_flexible_work"] = jobs_df["flexible_work_arrangements"].apply(
    lambda x: isinstance(x, list) and len(x) > 0
)

In [409]:
jobs_df = jobs_df.drop(columns=["flexible_work_arrangements"], errors="ignore")

In [410]:
jobs_df["ssoc_3d"] = jobs_df["ssoc_code"].astype(str).str[:3]

In [411]:
jobs_df = jobs_df.drop(columns=[
    "ssoc_code",
    "ssoc_version",
    "description",
    "description_clean",
    "title",
    "title_clean",
    "position_levels",
    "minimum_years_experience",
    "salary_type"
])

## Employment Type

In [412]:
contract_types = [
    "Permanent", "Contract", "Temporary", "Freelance"
]

work_types = [
    "Full Time", "Part Time"
]

def extract_contract_type(lst):
    if isinstance(lst, list):
        for item in lst:
            if item in contract_types:
                return item
    return None

def extract_work_type(lst):
    if isinstance(lst, list):
        for item in lst:
            if item in work_types:
                return item
    return None

jobs_df["contract_type"] = jobs_df["employment_types"].apply(extract_contract_type)
jobs_df["work_type"] = jobs_df["employment_types"].apply(extract_work_type)

jobs_df = jobs_df.drop(columns=["employment_types"])

In [413]:
jobs_df["contract_type"] = jobs_df["contract_type"].fillna("Unknown")
jobs_df["work_type"] = jobs_df["work_type"].fillna("Unknown")

In [414]:
jobs_df = jobs_df[
    ~(
        (jobs_df["contract_type"] == "Unknown") &
        (jobs_df["work_type"] == "Unknown")
    )
]

In [415]:
jobs_df["work_type"].value_counts()

work_type
Full Time    6894
Unknown      1287
Part Time     766
Name: count, dtype: int64

## Filling up work_type Unknown data

In [416]:
jobs_df["avg_salary"] = (
    jobs_df["salary_minimum"] + jobs_df["salary_maximum"]
) / 2

In [417]:
jobs_df.groupby("work_type")["avg_salary"].describe()

,count,mean,std,min,25%,50%,75%,max
work_type,,,,,,,,
Full Time,6894.0,3548.176168,1306.335695,1.0,2750.0,3400.0,4000.0,22500.0
Part Time,766.0,2773.075718,2205.739022,1.5,1600.0,2500.0,3750.0,24000.0
Unknown,1287.0,3269.283605,1724.239585,1.0,2395.0,2950.0,3750.0,16499.5


In [418]:
jobs_df.groupby("ssoc_3d")["work_type"].value_counts(normalize=True)

ssoc_3d  work_type
111      Full Time    0.875000
         Unknown      0.083333
         Part Time    0.041667
112      Full Time    0.956522
         Unknown      0.043478
                        ...   
941      Unknown      0.074074
961      Full Time    1.000000
962      Full Time    0.660714
         Part Time    0.214286
         Unknown      0.125000
Name: proportion, Length: 285, dtype: float64

In [419]:
jobs_df.groupby("work_type")["avg_salary"].median()

work_type
Full Time    3400.0
Part Time    2500.0
Unknown      2950.0
Name: avg_salary, dtype: float64

In [420]:
jobs_df.groupby("work_type")["avg_salary"].mean()

work_type
Full Time    3548.176168
Part Time    2773.075718
Unknown      3269.283605
Name: avg_salary, dtype: float64

In [421]:
# -------------------------
# Step 1: SSOC majority mapping
# -------------------------
ssoc_mode = (
    jobs_df[jobs_df["work_type"] != "Unknown"]
    .groupby("ssoc_3d")["work_type"]
    .agg(lambda x: x.mode()[0])
)

# -------------------------
# Step 2: Salary threshold (from your data)
# -------------------------
# midpoint between Full Time and Part Time medians
salary_medians = jobs_df.groupby("work_type")["avg_salary"].median()

full_time_median = salary_medians.get("Full Time")
part_time_median = salary_medians.get("Part Time")

# midpoint between the two
salary_threshold = (full_time_median + part_time_median) / 2

# -------------------------
# Step 3: Imputation function
# -------------------------
def impute_work_type(row):
    if row["work_type"] != "Unknown":
        return row["work_type"]
    
    # try SSOC first
    if row["ssoc_3d"] in ssoc_mode:
        return ssoc_mode[row["ssoc_3d"]]
    
    # fallback: salary
    if row["avg_salary"] >= salary_threshold:
        return "Full Time"
    else:
        return "Part Time"


# -------------------------
# Step 4: Apply
# -------------------------
jobs_df["work_type"] = jobs_df.apply(impute_work_type, axis=1)

In [422]:
jobs_df["work_type"].value_counts()

work_type
Full Time    8164
Part Time     783
Name: count, dtype: int64

In [423]:
jobs_df.groupby("work_type")["avg_salary"].median()

work_type
Full Time    3400.0
Part Time    2500.0
Name: avg_salary, dtype: float64

In [424]:
jobs_df.groupby("work_type")["avg_salary"].mean()

work_type
Full Time    3505.988670
Part Time    2771.365262
Name: avg_salary, dtype: float64

In [425]:
jobs_df["contract_type"].value_counts()

contract_type
Unknown      4079
Permanent    3557
Contract      882
Temporary     361
Freelance      68
Name: count, dtype: int64

# Job Skills

## Raw Job Skills

In [426]:
def clean_skill(skill):
    if not isinstance(skill, str):
        return None

    skill = skill.lower().strip()
    return skill


jobs_df["skills_clean"] = jobs_df["skills"].apply(
    lambda lst: list(set(
        clean_skill(s) for s in lst if clean_skill(s)
    ))
)

jobs_df = jobs_df.drop(columns=["skills"])

In [427]:
from collections import Counter

all_skills = [
    skill
    for lst in jobs_df["skills_clean"]
    for skill in lst
]

skill_counts = Counter(all_skills)

skills_df = (
    pd.DataFrame(skill_counts.items(), columns=["skill", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

In [428]:
skills_df.to_excel("../../data/job_skills_raw.xlsx", index=False)

In [429]:
jobs_df["skills_clean"] = jobs_df["skills"].apply(
    lambda lst: list(set(
        clean_skill(s) for s in lst if clean_skill(s)
    ))
)

KeyError: 'skills'

In [ ]:
from difflib import SequenceMatcher

def is_similar(a, b, threshold=0.8):
    return SequenceMatcher(None, a, b).ratio() > threshold


def dedupe_similar_skills(skill_list):
    unique = []
    
    for skill in skill_list:
        if not any(is_similar(skill, existing) for existing in unique):
            unique.append(skill)
    
    return unique

In [ ]:
jobs_df["skills_clean"] = jobs_df["skills_clean"].apply(dedupe_similar_skills)

In [ ]:
from collections import Counter

freq_counts = Counter(skill_counts.values())

In [ ]:
freq_df = (
    pd.DataFrame(freq_counts.items(), columns=["frequency", "num_skills"])
    .sort_values("frequency")
    .reset_index(drop=True)
)

freq_df

,frequency,num_skills
0,1,2006
1,2,854
2,3,465
3,4,345
4,5,254
...,...,...
270,2048,1
271,2339,1
272,2710,1
273,2808,1


## Cleaning of Job Skills

In [ ]:
import re
from collections import Counter
import pandas as pd

core_soft_skills = [
    "communication",
    "leadership",
    "presentation",
    "interpersonal",
    "customer service",
    "problem solving",
    "analytical",
    "teamwork",
    "independence",
]

exact_keep = {
    "project management",
    "management accounting",
    "financial management",
    "risk management",
    "data management",
}

junk_skills = {
    "team player",
    "able to work independently",
    "physically fit",
}

def clean_skill(skill):
    if not isinstance(skill, str):
        return None

    skill = skill.lower().strip()
    skill = skill.replace("&", "and")
    skill = re.sub(r"[^\w\s]", "", skill)
    skill = re.sub(r"\s+", " ", skill).strip()

    if skill in junk_skills:
        return None

    if skill in exact_keep:
        return skill

    for core in core_soft_skills:
        if core in skill:
            return core

    skill = re.sub(r"\bskills\b", "", skill)
    skill = re.sub(r"\s+", " ", skill).strip()

    return skill if skill else None

# 1. clean + dedupe within each row USING skills_clean
jobs_df["skills_clean"] = jobs_df["skills_clean"].apply(
    lambda lst: list(set(
        clean_skill(s) for s in lst if clean_skill(s)
    )) if isinstance(lst, list) else []
)

# 2. count skill frequencies AFTER cleaning
all_skills = [
    skill
    for lst in jobs_df["skills_clean"]
    for skill in lst
]

skill_counts = Counter(all_skills)

# 3. keep only skills with frequency >= 3
valid_skills = {
    skill for skill, count in skill_counts.items()
    if count >= 3
}

jobs_df["skills_clean"] = jobs_df["skills_clean"].apply(
    lambda lst: [s for s in lst if s in valid_skills]
)

# 4. check rows that became empty
jobs_df["num_skills"] = jobs_df["skills_clean"].apply(len)

print(jobs_df["num_skills"].describe())
print("Empty rows:", (jobs_df["num_skills"] == 0).sum())

# 5. inspect final skill frequencies
final_skill_counts = Counter(
    skill for lst in jobs_df["skills_clean"] for skill in lst
)

skills_df = (
    pd.DataFrame(final_skill_counts.items(), columns=["skill", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

skills_df.head(10)

count    8947.000000
mean       13.040684
std         3.390838
min         1.000000
25%        10.000000
50%        13.000000
75%        15.000000
max        20.000000
Name: num_skills, dtype: float64
Empty rows: 0


,skill,count
0,communication,3384
1,customer service,2958
2,interpersonal,2830
3,leadership,2108
4,microsoft office,1934
5,sales,1744
6,microsoft excel,1732
7,inventory,1685
8,marketing,1573
9,administration,1350


# Save output

In [307]:
jobs_df.head(1)

,uuid,salary_minimum,salary_maximum,posting_date,expiry_date,has_flexible_work,ssoc_3d,skills_clean,contract_type,work_type,avg_salary
0,2cc5117f205f59b400cb529d2253651a,3000,6000,2026-01-29,2026-02-28,False,243,"[project bidding, technical demonstrations, sh...",Permanent,Full Time,4500.0


In [308]:
print("Remaining:", len(jobs_df))

Remaining: 8947


In [309]:
jobs_df.to_pickle("../../data/jobs_cleaned.pkl")